In [ ]:
%load_ext autoreload
%autoreload 2

import scanpy as sc

adata = sc.read_h5ad("./data/B_cell/bcell_velocity_standard.h5ad")
adata

In [ ]:
import scvelo as scv
import scanpy as sc
import anndata
import numpy as np
import sys
import numba, loompy

print("python:", sys.version)
print("scvelo:", scv.__version__)
print("scanpy:", sc.__version__)
print("anndata:", anndata.__version__)
print("numpy:", np.__version__)
print("numba:", numba.__version__)
print("loompy:", loompy.__version__)

In [ ]:
import flowmap
from flowmap import *
import pickle

with open("./data/B_cell/flowmap_emb.pkl", "rb") as f:
    emb = pickle.load(f)

emb

In [ ]:
import scvelo as scv

# Optional: re-compute neighbors if unsure
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# Leiden clustering
sc.tl.leiden(adata, resolution=0.6)  # adjust later if needed
adata.obsm["X_flowmap"] = emb.X_emb

scv.pl.velocity_embedding_stream(
    adata,
    basis="X_flowmap",
    color="leiden"
)

In [ ]:
markers = {
    "Naive": ["GRASP", "ZNF331", "IRS2", "LIX1-AS1", "NR4A2", "KCNH8", "DUSP1", "RASGEF1B", "FOSB", "ZBTB10"],
    "ActB": ["ITGA1", "JAZF1", "SMIM14", "AC083837.1", "ST6GALNAC3", "IL7", "NIBAN3"],
    "preGCBC": ["KIAA1549L", "CFI", "HOMER2", "EEPD1", "PALLD", "SLC37A3", "FCER2"],
    "prePB": ["IL2RB", "IL2RA", "CD226", "DUSP4", "ACSL4", "CLIC5", "IL12RB2"],
    "PB": ["CFAP54", "AL591518.1", "ACOXL", "FNDC3B", "AC016074.2", "NUGGC", "RASSF6", "ZNF215"]
}

markers_filtered = {}

for key, gene_list in markers.items():
    genes_present = [g for g in gene_list if g in adata.var_names]
    if len(genes_present) > 0:
        markers_filtered[key] = genes_present

markers_filtered

In [ ]:
import scanpy as sc
sc.pl.dotplot(
    adata,
    markers_filtered,
    groupby='leiden',
    standard_scale='var'
)

In [ ]:
# mapping dict
cluster_map = {
    "0": "prePB",
    "1": "PreGCBC",
    "2": "PB",
    "3": "ActB",
    "4": "ActB",
    "5": "ActB",
    "6": "PB",
    "7": "Naive",
    "8": "Naive",
    "9": "Naive",
    "10": "Naive",
    "11": "PreGCBC",
    "12": "Naive",
}

# assign new labels
adata.obs["celltype"] = adata.obs["leiden"].map(cluster_map)
adata.write("./data/B_cell/bcell_velocity_standard.h5ad")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.neighbors import NearestNeighbors
from flowmap.utils import compute_velocity_on_grid


# ------------------------------------------------------------
# Helper: remove grid seeds outside the manifold
# ------------------------------------------------------------
def points_inside_mask(X_emb, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X_emb)
    r = np.median(nn.kneighbors(X_emb)[0][:, -1]) * radius_scale
    neigh_idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh_idx])


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
X_emb = emb.X_emb
celltypes = adata.obs["celltype"].astype("category")
codes = celltypes.cat.codes
categories = celltypes.cat.categories

# ------------------------------------------------------------
# Velocity grid (sparser)
# ------------------------------------------------------------
Xg, keep_mass, Vg = compute_velocity_on_grid(
    X_emb,
    spline_vf=emb.spline_vf,
    grid_size=20,          # ↓ sparser grid
    min_mass=0.01
)

keep_inside = points_inside_mask(X_emb, Xg)
Xg = Xg[keep_inside]
Vg = Vg[keep_inside]


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 8))
# cmap = plt.get_cmap("tab20", len(categories))  # good for up to ~20 types
colors = plt.get_cmap("Set2").colors  # soft + distinct
color_list = [colors[i % len(colors)] for i in range(len(categories))]
# scatter (larger + more transparent)
sc = ax.scatter(
    X_emb[:, 0],
    X_emb[:, 1],
    # c=velocity_pseudotime,
    # cmap="viridis",
    # c=codes,
    c=[color_list[i] for i in codes],
    # cmap=cmap,
    s=100,
    alpha=0.1,
    linewidths=0,
)

# velocity arrows (thicker + bigger heads)
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=7,
    width=0.005,         # ↑ thicker
    headwidth=6.0,       # ↑ bigger head
    headlength=6.0,
    headaxislength=4.0,
    minlength=0.2,
    color="k",
    alpha=0.9,
)

# for i, cat in enumerate(categories):
#     mask = (codes == i)
#     if np.sum(mask) == 0:
#         continue
    
#     x_mean = X_emb[mask, 0].mean()
#     y_mean = X_emb[mask, 1].mean()
    
#     ax.text(
#         x_mean,
#         y_mean,
#         cat,
#         fontsize=20,
#         weight="bold",
#         ha="center",
#         va="center",
#         color="black",
#         bbox=dict(
#             boxstyle="round,pad=0.2",
#             facecolor="white",
#             alpha=0.7,
#             linewidth=0
#         )
#     )
    
# ------------------------------------------------------------
# Styling (no legend / no colorbar)
# ------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
legend_elements = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        label=cat,
        markerfacecolor=color_list[i],
        markersize=20
    )
    for i, cat in enumerate(categories)
]

ax.legend(
    handles=legend_elements,
    fontsize=15,
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False,
)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig(
    "./figures/bcell/bcell_velocity_stream.pdf",  # updated path
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from flowmap.geometry.curvature import compute_flow_curvature

curv = compute_flow_curvature(emb)

In [ ]:
curv

In [ ]:
X = emb.X_emb

k_total  = curv["curvature"]["total"]
k_steer   = curv["curvature"]["steer"]
k_surface = curv["curvature"]["surface"]

def clip_quantile(arr, q_low=2, q_high=98):
    lo, hi = np.percentile(arr, [q_low, q_high])
    return np.clip(arr, lo, hi)

k_total_c  = clip_quantile(k_total)
k_steer_c   = clip_quantile(k_steer)
k_surface_c = clip_quantile(k_surface)

fig, axes = plt.subplots(1,3, figsize=(18,5))

axes[0].scatter(X[:,0], X[:,1], c=k_total_c, s=6, cmap="coolwarm")
axes[0].set_title("Total curvature")

axes[1].scatter(X[:,0], X[:,1], c=k_steer_c, s=6, cmap="coolwarm")
axes[1].set_title("Geodesic curvature")

axes[2].scatter(X[:,0], X[:,1], c=k_surface_c, s=6, cmap="coolwarm")
axes[2].set_title("Normal curvature")

for ax in axes:
    ax.set_aspect("equal")
    ax.axis("off")

plt.savefig("./figures/bcell/curvature_panels.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle


fig, ax = plt.subplots(figsize=(8, 8))

# ------------------------------------------------------------
# Velocity grid (sparser)
# ------------------------------------------------------------
Xg, keep_mass, Vg = compute_velocity_on_grid(
    X_emb,
    spline_vf=emb.spline_vf,
    grid_size=18,          # ↓ sparser grid
    min_mass=0.01
)

keep_inside = points_inside_mask(X_emb, Xg)
Xg = Xg[keep_inside]
Vg = Vg[keep_inside]


# ------------------------------------------------------------
# Colormap normalization (shift white downward)
# ------------------------------------------------------------
vmin = np.percentile(k_steer_c, 2)
vmax = np.percentile(k_steer_c, 98)

# shift center slightly negative → more red overall
norm = TwoSlopeNorm(
    vmin=vmin,
    vcenter=0.2,   # 👈 adjust this (more negative = more red bias)
    vmax=vmax
)

# ------------------------------------------------------------
# Highlight region (your box)
# ------------------------------------------------------------
x1, x2 = 4.5, 10.0
y1, y2 = -3.0, 2.0

# ------------------------------------------------------------
# Define mask for highlighted region
# ------------------------------------------------------------
mask_box = (
    (X_emb[:, 0] > x1) & (X_emb[:, 0] < x2) &
    (X_emb[:, 1] > y1) & (X_emb[:, 1] < y2)
)

# ------------------------------------------------------------
# Background scatter (bottom)
# ------------------------------------------------------------
sc = ax.scatter(
    X_emb[:, 0],
    X_emb[:, 1],
    c=k_steer_c,
    cmap="coolwarm",
    norm=norm,
    s=8,
    alpha=0.65,
    linewidths=0,
    zorder=1
)

# ------------------------------------------------------------
# Optional highlight (still scatter layer)
# ------------------------------------------------------------
ax.scatter(
    X_emb[mask_box, 0],
    X_emb[mask_box, 1],
    c=k_steer_c[mask_box],
    cmap="coolwarm",
    norm=norm,
    s=10,
    alpha=0.9,
    linewidths=0,
    zorder=2
)

# ------------------------------------------------------------
# Rectangle (middle)
# ------------------------------------------------------------
# rect = Rectangle(
#     (x1, y1),
#     x2 - x1,
#     y2 - y1,
#     linewidth=4.5,
#     edgecolor="#2ca02c",
#     facecolor="none",
#     linestyle="--",
#     zorder=3
# )
# ax.add_patch(rect)

# ------------------------------------------------------------
# Velocity arrows (top layer)
# ------------------------------------------------------------
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=7,
    width=0.005,
    headwidth=6.0,
    headlength=6.0,
    headaxislength=4.0,
    minlength=0.2,
    color="k",
    alpha=0.9,
    zorder=4
)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()

plt.savefig(
    "./figures/bcell/bcell_curvature_velocity_stream.pdf",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
vmin, vmax

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle


fig, ax = plt.subplots(figsize=(8, 8))

# ------------------------------------------------------------
# Velocity grid (sparser)
# ------------------------------------------------------------
Xg, keep_mass, Vg = compute_velocity_on_grid(
    X_emb,
    spline_vf=emb.spline_vf,
    grid_size=18,          # ↓ sparser grid
    min_mass=0.01
)

keep_inside = points_inside_mask(X_emb, Xg)
Xg = Xg[keep_inside]
Vg = Vg[keep_inside]


# ------------------------------------------------------------
# Colormap normalization (shift white downward)
# ------------------------------------------------------------
vmin = np.percentile(k_surface_c, 2)
vmax = np.percentile(k_surface_c, 98)

# shift center slightly negative → more red overall
norm = TwoSlopeNorm(
    vmin=vmin,
    vcenter=0.4,
    vmax=vmax
)

# ------------------------------------------------------------
# Highlight region (your box)
# ------------------------------------------------------------
x1, x2 = 4.5, 10.0
y1, y2 = -3.0, 2.0

# ------------------------------------------------------------
# Define mask for highlighted region
# ------------------------------------------------------------
mask_box = (
    (X_emb[:, 0] > x1) & (X_emb[:, 0] < x2) &
    (X_emb[:, 1] > y1) & (X_emb[:, 1] < y2)
)

# ------------------------------------------------------------
# Background scatter (bottom)
# ------------------------------------------------------------
sc = ax.scatter(
    X_emb[:, 0],
    X_emb[:, 1],
    c=k_surface_c,
    cmap="coolwarm",
    norm=norm,
    s=8,
    alpha=0.65,
    linewidths=0,
    zorder=1
)

# ------------------------------------------------------------
# Optional highlight (still scatter layer)
# ------------------------------------------------------------
ax.scatter(
    X_emb[mask_box, 0],
    X_emb[mask_box, 1],
    c=k_surface_c[mask_box],
    cmap="coolwarm",
    norm=norm,
    s=10,
    alpha=0.9,
    linewidths=0,
    zorder=2
)

# ------------------------------------------------------------
# Rectangle (middle)
# ------------------------------------------------------------
# rect = Rectangle(
#     (x1, y1),
#     x2 - x1,
#     y2 - y1,
#     linewidth=4.5,
#     edgecolor="#2ca02c",
#     facecolor="none",
#     linestyle="--",
#     zorder=3
# )
# ax.add_patch(rect)

# ------------------------------------------------------------
# Velocity arrows (top layer)
# ------------------------------------------------------------
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=7,
    width=0.005,
    headwidth=6.0,
    headlength=6.0,
    headaxislength=4.0,
    minlength=0.2,
    color="k",
    alpha=0.9,
    zorder=4
)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()

plt.savefig(
    "./figures/bcell/bcell_surface_curvature_velocity_stream.pdf",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ------------------------------------------------------------
# Region (same as before)
# ------------------------------------------------------------
x_min, x_max = 12.0, 17.0
y_min, y_max = 6.5, 10.0

coords = emb.X_emb
vals   = k_geod_c

mask = (
    (coords[:, 0] > x_min) & (coords[:, 0] < x_max) &
    (coords[:, 1] > y_min) & (coords[:, 1] < y_max)
)

coords_zoom = coords[mask]
vals_zoom   = vals[mask]


# ------------------------------------------------------------
# Plot base
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 7))

sc = ax.scatter(
    coords_zoom[:, 0],
    coords_zoom[:, 1],
    c=vals_zoom,
    cmap="coolwarm",
    s=100,
    alpha=0.9,
    linewidths=0,
)


# ------------------------------------------------------------
# Styling (with grid for reference)
# ------------------------------------------------------------
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

# ax.set_xticks(np.linspace(x_min, x_max, 6))
# ax.set_yticks(np.linspace(y_min, y_max, 6))
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# ax.set_xlabel("UMAP1")
# ax.set_ylabel("UMAP2")

ax.set_aspect("equal")
plt.tight_layout()

plt.savefig(
    "./figures/bcell/curvature_region.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ------------------------------------------------------------
# Region (same as before)
# ------------------------------------------------------------
x_min, x_max = 12.0, 17.0
y_min, y_max = 6.5, 10.0

coords = emb.X_emb
vals   = k_geod_c

mask = (
    (coords[:, 0] > x_min) & (coords[:, 0] < x_max) &
    (coords[:, 1] > y_min) & (coords[:, 1] < y_max)
)

coords_zoom = coords[mask]
vals_zoom   = vals[mask]


# ------------------------------------------------------------
# Simple rule-based clustering
# ------------------------------------------------------------
curv_thresh = 0.3

# line: y = x + 3
mask_high = vals_zoom > curv_thresh

labels = -1 * np.ones(len(coords_zoom), dtype=int)

# direction vector
p1 = np.array([13, 8.4])
p2 = np.array([17, 7])
d = p2 - p1

# normal vector
n = np.array([-d[1], d[0]])

# signed distance
signed_dist = (coords_zoom - p1) @ n

labels[(signed_dist > 0) & mask_high] = 0
labels[(signed_dist <= 0) & mask_high] = 1

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))

# background (low curvature or outside rule)
ax.scatter(
    coords_zoom[labels == -1, 0],
    coords_zoom[labels == -1, 1],
    c="lightgray",
    s=15,
    alpha=0.3,
    linewidths=0,
    zorder=1
)

# clusters
ax.scatter(
    coords_zoom[labels != -1, 0],
    coords_zoom[labels != -1, 1],
    c=labels[labels != -1],
    cmap="tab10",
    s=30,
    alpha=0.9,
    linewidths=0,
    zorder=2
)

# draw line y = x + 3
x_line = np.linspace(x_min, x_max, 200)
y_line = 1.05 * x_line + 3

ax.plot(
    x_line,
    y_line,
    color="black",
    linewidth=2.5,
    linestyle="-",
    alpha=0.9,
)

# styling
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

ax.set_aspect("equal")
# ax.set_xticks([]); ax.set_yticks([])
# for spine in ax.spines.values():
#     spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc
import pandas as pd

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
adata_raw = ad.read_h5ad("./data/B_cell/IGVFFI3928IUMP.h5ad")


# ------------------------------------------------------------
# Basic filtering (cells only, KEEP ALL GENES)
# ------------------------------------------------------------
sc.pp.filter_cells(adata_raw, min_counts=1000)
sc.pp.filter_cells(adata_raw, min_genes=500)


# ------------------------------------------------------------
# Set layers for consistency (important if using velocity data)
# ------------------------------------------------------------
if "mature" in adata_raw.layers:
    adata_raw.layers["spliced"] = adata_raw.layers["mature"]
if "nascent" in adata_raw.layers:
    adata_raw.layers["unspliced"] = adata_raw.layers["nascent"]

if "ambiguous" in adata_raw.layers:
    del adata_raw.layers["ambiguous"]


# ------------------------------------------------------------
# Normalize + log (REQUIRED for DEG)
# ------------------------------------------------------------
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)


# ------------------------------------------------------------
# Ensembl → gene symbol (mygene)
# ------------------------------------------------------------
import mygene

mg = mygene.MyGeneInfo()

original_ids = adata_raw.var_names.tolist()
clean_ids = [str(g).split(".")[0] for g in original_ids]

print("Mapping genes...")

res = mg.querymany(
    clean_ids,
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    as_dataframe=False,
    verbose=False
)

symbol_map = {}
for item in res:
    q = item.get("query")
    s = item.get("symbol")
    if q and s and str(s).lower() != "nan":
        symbol_map[q] = str(s)

# assign symbols (fallback = Ensembl ID)
new_names = []
for i, orig in enumerate(original_ids):
    clean = clean_ids[i]
    new_names.append(symbol_map.get(clean, clean))

adata_raw.var["original_id"] = original_ids
adata_raw.var_names = new_names
adata_raw.var_names_make_unique()

print("Gene mapping done.")


# ------------------------------------------------------------
# OPTIONAL sanity check
# ------------------------------------------------------------
print(adata_raw)
print("Total genes:", adata_raw.n_vars)

In [ ]:
# ============================================================
# Region mask (ONLY to map labels back)
# ============================================================
coords = emb.X_emb

x_min, x_max = 12.0, 17.0
y_min, y_max = 6.5, 10.0

mask_region = (
    (coords[:, 0] > x_min) & (coords[:, 0] < x_max) &
    (coords[:, 1] > y_min) & (coords[:, 1] < y_max)
)

region_indices = np.where(mask_region)[0]

# ============================================================
# Use EXISTING labels (important)
# ============================================================
# labels must already exist for coords_zoom
# e.g. labels.shape == number of cells in region

labels_full = -1 * np.ones(adata_raw.n_obs, dtype=int)
labels_full[region_indices] = labels   # <-- use your existing labels

adata_raw.obs["geom_cluster"] = labels_full


# ============================================================
# Subset to clusters 0 / 1
# ============================================================
mask = np.isin(adata_raw.obs["geom_cluster"], [0, 1])
adata_sub = adata_raw[mask].copy()

if adata_raw.raw is not None:
    adata_sub.raw = adata_raw.raw[mask].copy()
else:
    adata_sub.raw = adata_sub

adata_sub.obs["geom_cluster"] = (
    adata_sub.obs["geom_cluster"].astype(str).astype("category")
)


# ============================================================
# DEG
# ============================================================
sc.tl.rank_genes_groups(
    adata_sub,
    groupby="geom_cluster",
    groups=["0"],
    reference="1",
    method="wilcoxon",
)


# ============================================================
# Extract results
# ============================================================
de = adata_sub.uns["rank_genes_groups"]

deg_df = pd.DataFrame({
    "gene": de["names"]["0"],
    "score": de["scores"]["0"],
    "logFC": de["logfoldchanges"]["0"],
    "pval": de["pvals"]["0"],
    "pval_adj": de["pvals_adj"]["0"],
}).sort_values("pval_adj")

deg_df.head(20)

In [ ]:
alpha = 0.05
logfc_cut = 0.5

# ------------------------------------------------
# Prepare data
# ------------------------------------------------
df_clean = deg_df.dropna(subset=["logFC", "pval_adj"]).copy()
df_clean["-log10p"] = -np.log10(df_clean["pval_adj"].clip(lower=1e-300))

sig_both = (df_clean["pval_adj"] < alpha) & (df_clean["logFC"].abs() >= logfc_cut)

# pick top genes (set N_annotate = 0 to disable)
N_annotate = 8
df_sig_sorted = (
    df_clean.loc[sig_both]
             .sort_values("pval_adj")
             .head(N_annotate)
)

# ------------------------------------------------
# PLOT
# ------------------------------------------------
plt.figure(figsize=(6., 7.8))

# background
plt.scatter(
    df_clean.loc[~sig_both, "logFC"],
    df_clean.loc[~sig_both, "-log10p"],
    s=140, c="#C7C7C7", alpha=0.55, edgecolors="none"
)

# significant points
plt.scatter(
    df_clean.loc[sig_both, "logFC"],
    df_clean.loc[sig_both, "-log10p"],
    s=240, c="#B22222", alpha=0.9,
    edgecolors="black", linewidth=0.25
)

# cutoff lines
plt.axhline(-np.log10(alpha), color="black", linestyle="--", lw=1)
plt.axvline(logfc_cut, color="black", linestyle="--", lw=1)
plt.axvline(-logfc_cut, color="black", linestyle="--", lw=1)

# ------------------------------------------------
# >>> OPTIONAL ANNOTATION (comment this entire block out) <<<
# ------------------------------------------------
if N_annotate > 0:
    for _, row in df_sig_sorted.iterrows():
        plt.text(
            row["logFC"] + 0.10,
            row["-log10p"] + 0.10,
            row["gene"],
            fontsize=18,
            ha="left",
            va="bottom"
        )

# ------------------------------------------------
# Aesthetics
# ------------------------------------------------
ax = plt.gca()

# keep only axis lines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["bottom", "left"]:
    ax.spines[spine].set_linewidth(1.3)

# labels
plt.xlabel(r"log$_2$ FC (a / b)", fontsize=28)
plt.ylabel(r"-log$_{10}$(adjusted p)", fontsize=28)

# ticks
plt.yticks([0, 12, 24], fontsize=20)
plt.xticks([-1.6, -0.8, 0.0, 0.8, 1.6], fontsize=20)

plt.xlim(-1.6, 1.6)
plt.grid(False)
plt.tick_params(axis="both", length=5, width=1.2, color="black")

plt.tight_layout()

# ------------------------------------------------
# SAVE (PDF vector format)
# ------------------------------------------------
plt.savefig(
    "./figures/bcell/bcell_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import scanpy as sc
import scvelo as scv
import pandas as pd

# ============================================================
# Load data
# ============================================================
adata = sc.read_h5ad("./data/B_cell/bcell_velocity_standard.h5ad")

# basic setup
scv.settings.verbosity = 3
scv.settings.set_figure_params("scvelo")


# ============================================================
# Preprocessing
# ============================================================
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)


# ============================================================
# Velocity
# ============================================================
scv.tl.velocity(adata, mode="stochastic")
scv.tl.velocity_graph(adata)


# ============================================================
# Pseudotime (velocity-based)
# ============================================================
scv.tl.velocity_pseudotime(adata, n_dcs=10)

# visualize
scv.pl.scatter(adata, color="velocity_pseudotime", cmap="viridis")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Pseudotime
# ------------------------------------------------------------
pseudotime = adata.obs["velocity_pseudotime"].values

vmin = np.percentile(pseudotime, 20)
vmax = np.percentile(pseudotime, 80)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 8))

sc = ax.scatter(
    X_emb[:, 0],
    X_emb[:, 1],
    c=pseudotime,
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    s=10,
    alpha=0.8,
    linewidths=0
)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
# ax.set_aspect("equal")
# ax.set_xticks([])
# ax.set_yticks([])

# for spine in ax.spines.values():
#     spine.set_visible(False)

# optional colorbar
cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Velocity pseudotime")

plt.tight_layout()

plt.savefig(
    "./figures/bcell/bcell_velocity_pseudotime.pdf",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

X = emb.X_emb

genes = ["IRF4", "PRDM1"]  # BLIMP1 = PRDM1

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, gene in enumerate(genes):
    
    if gene not in adata.var_names:
        print(f"{gene} not found!")
        continue
    
    # extract expression
    expr = adata[:, gene].X
    
    # handle sparse
    if hasattr(expr, "toarray"):
        expr = expr.toarray().flatten()
    else:
        expr = np.array(expr).flatten()
    
    # optional: clip for visualization (same trick as curvature)
    lo, hi = np.percentile(expr, [2, 98])
    expr = np.clip(expr, lo, hi)

    sc = axes[i].scatter(
        X[:, 0],
        X[:, 1],
        c=expr,
        cmap="viridis",
        s=6,
        alpha=0.9,
        linewidths=0,
    )
    
    axes[i].set_title(gene)
    axes[i].set_aspect("equal")
    axes[i].axis("off")
    
    # plt.colorbar(sc, ax=axes[i], fraction=0.046, pad=0.04)

plt.tight_layout()

os.makedirs("./figures/bcell/", exist_ok=True)

plt.savefig(
    "./figures/bcell/irf4_prdm1_expression.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
%load_ext autoreload
%autoreload 2

from flowmap.geometry import LagrangianPathOptimizer
import flowmap


path_init = np.array([
    [5.0, 11.0],
    [15.0, 7.8],
    [18.0, 6.7],
])

def resample_polyline(points, n_points=100):
    points = np.asarray(points)

    # segment lengths
    segs = np.linalg.norm(points[1:] - points[:-1], axis=1)
    cumlen = np.concatenate([[0.0], np.cumsum(segs)])
    total_len = cumlen[-1]

    t_new = np.linspace(0, total_len, n_points)

    new_pts = []
    for t in t_new:
        idx = np.searchsorted(cumlen, t) - 1
        idx = np.clip(idx, 0, len(segs) - 1)

        t0, t1 = cumlen[idx], cumlen[idx + 1]
        p0, p1 = points[idx], points[idx + 1]

        w = 0 if t1 == t0 else (t - t0) / (t1 - t0)
        new_pts.append((1 - w) * p0 + w * p1)

    return np.array(new_pts)

path_init = resample_polyline(path_init, n_points=100)

lap = LagrangianPathOptimizer(emb)

res = lap.fit_path(
    path_init=path_init,
    distance_mode="orig",
    subsample_n=4000,
    k=20,
    alpha=0.0,
    n_segments=100,
    lr=5e-3,
    iters=300,
)

refined_paths = res["path_refined"]

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))

# ------------------------------------------------------------
# Background
# ------------------------------------------------------------
plt.scatter(
    X_emb[:, 0], X_emb[:, 1],
    s=5,
    alpha=0.2,
    c=k_geod_c,
    cmap="coolwarm"
)

# ------------------------------------------------------------
# Ensure correct shape
# ------------------------------------------------------------
p_init = path_init

p_ref = refined_paths
if p_ref.ndim == 3:   # if wrapped in list
    p_ref = p_ref[0]

# ------------------------------------------------------------
# Plot paths
# ------------------------------------------------------------

# initial (dashed)
plt.plot(
    p_init[:, 0], p_init[:, 1],
    "--",
    color="black",
    alpha=0.6,
    label="initial"
)

# refined (solid)
plt.plot(
    p_ref[:, 0], p_ref[:, 1],
    "-",
    color="red",
    linewidth=2.5,
    label="refined"
)

# start / end
plt.scatter(p_ref[0, 0], p_ref[0, 1], color="green", s=80, marker="o", label="start")
plt.scatter(p_ref[-1, 0], p_ref[-1, 1], color="black", s=80, marker="x", label="end")

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
plt.gca().set_aspect("equal")
plt.xticks([])
plt.yticks([])
plt.legend(frameon=False)
plt.title("Least-action path (refined)")

for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

fig, ax = plt.subplots(figsize=(8, 8))

# ------------------------------------------------------------
# Colormap normalization (curvature)
# ------------------------------------------------------------
vmin = np.percentile(k_geod_c, 2)
vmax = np.percentile(k_geod_c, 98)

norm = TwoSlopeNorm(
    vmin=vmin,
    vcenter=0.2,
    vmax=vmax
)

# ------------------------------------------------------------
# Background: curvature landscape
# ------------------------------------------------------------
sc = ax.scatter(
    X_emb[:, 0],
    X_emb[:, 1],
    c=k_geod_c,
    cmap="coolwarm",
    norm=norm,
    s=12,
    alpha=0.45,
    linewidths=0,
    zorder=1
)

# ------------------------------------------------------------
# Refined path
# ------------------------------------------------------------
p_ref = refined_paths
if p_ref.ndim == 3:
    p_ref = p_ref[0]

ax.plot(
    p_ref[:, 0], p_ref[:, 1],
    "-",
    color="#6a3d9a",
    linewidth=4.0,
    zorder=3
)

# start (clean, neutral)
ax.scatter(
    p_ref[0, 0], p_ref[0, 1],
    facecolor="white",
    edgecolor="black",
    linewidth=1.5,
    s=360,
    marker="o",
    zorder=4
)

# end (distinct, warm but not red)
ax.scatter(
    p_ref[-1, 0], p_ref[-1, 1],
    color="#f1c40f",   # gold
    edgecolor="black",
    linewidth=1.2,
    s=360,
    marker="o",
    zorder=4
)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

# colorbar
# cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Geodesic curvature")

plt.tight_layout()

plt.savefig(
    "./figures/bcell/bcell_curvature_with_path_clean.pdf",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Curvature along path (neighbor averaging)
# ------------------------------------------------------------
def curvature_along_path(path, X_emb, k_geod_c, epsilon=0.05):
    scale = np.mean(np.ptp(X_emb, axis=0))
    radius = epsilon * scale

    curv = np.empty(len(path))

    for i, p in enumerate(path):
        dists = np.linalg.norm(X_emb - p, axis=1)
        mask = dists <= radius

        curv[i] = np.mean(k_geod_c[mask]) if np.any(mask) else np.nan

    return curv


# ------------------------------------------------------------
# Prepare path
# ------------------------------------------------------------
p_ref = refined_paths
if p_ref.ndim == 3:
    p_ref = p_ref[0]

# ------------------------------------------------------------
# Compute curvature
# ------------------------------------------------------------
curv = curvature_along_path(p_ref, X_emb, k_geod_c, epsilon=0.05)

# handle NaNs (interpolate instead of zeroing — important)
valid = np.isfinite(curv)
curv = np.interp(np.arange(len(curv)), np.where(valid)[0], curv[valid])

# ------------------------------------------------------------
# Arc-length parameterization
# ------------------------------------------------------------
ds = np.linalg.norm(np.diff(p_ref, axis=0), axis=1)
s = np.concatenate([[0.0], np.cumsum(ds)])

# ------------------------------------------------------------
# Proper cumulative curvature: ∫ κ ds
# ------------------------------------------------------------
cum_curv = np.zeros_like(curv)
cum_curv[1:] = np.cumsum(
    0.5 * (curv[:-1] + curv[1:]) * ds
)

# optional normalization (for visualization only)
cum_curv_norm = cum_curv / (np.max(np.abs(cum_curv)) + 1e-8)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
plt.figure(figsize=(6, 4))

# curvature (original scale)
plt.plot(
    s, curv,
    color="#cc8c00",
    linewidth=2.0,
    linestyle=":",
    label="curvature"
)

# cumulative curvature (dominant)
plt.plot(
    s, cum_curv,
    color="black",
    linewidth=3.0,
    label="cumulative curvature"
)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
plt.xlabel("Arc length along path")
plt.ylabel("Curvature")
plt.title("Curvature along trajectory")

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ============================================================
# 0. Prepare gene matrix
# ============================================================
X_gene = adata_raw.X
if hasattr(X_gene, "toarray"):
    X_gene = X_gene.toarray()

gene_names = adata_raw.var_names.to_numpy()

# ============================================================
# 1. Path geometry (arc length)
# ============================================================
p_ref = refined_paths
if p_ref.ndim == 3:
    p_ref = p_ref[0]

ds = np.linalg.norm(np.diff(p_ref, axis=0), axis=1)
s_path = np.concatenate([[0.0], np.cumsum(ds)])

# ============================================================
# 2. Project cells → path coordinate
# ============================================================
def project_cells_to_path(X_emb, path, s_path):
    dists = np.linalg.norm(
        X_emb[:, None, :] - path[None, :, :],
        axis=2
    )
    idx = np.argmin(dists, axis=1)
    return s_path[idx]

s_cells = project_cells_to_path(X_emb, p_ref, s_path)

# ============================================================
# 3. Filter genes
# ============================================================
mean_expr = X_gene.mean(axis=0)
var_expr  = X_gene.var(axis=0)

mask = (mean_expr > 0.05) & (var_expr > 0.01)

Xf = X_gene[:, mask]
genes_f = gene_names[mask]

print(f"Using {Xf.shape[1]} genes")

# ============================================================
# 4. Kernel smoothing (cells → path)
# ============================================================
bandwidth = 0.05 * (s_path.max() - s_path.min())

diff = s_path[:, None] - s_cells[None, :]
W = np.exp(-0.5 * (diff / bandwidth)**2)
W /= (W.sum(axis=1, keepdims=True) + 1e-8)

# smoothed gene curves (path_points × genes)
G = W @ Xf

# ============================================================
# 5. Prepare regression target (curvature on path)
# ============================================================
# assume curv already computed + cleaned
valid = np.isfinite(curv)
curv_smooth = np.interp(s_path, s_path[valid], curv[valid])

# standardize y ONLY
y = (curv_smooth - curv_smooth.mean()) / (curv_smooth.std() + 1e-8)

# ============================================================
# 6. Linear regression along path (vectorized)
# ============================================================
G_centered = G - G.mean(axis=0)

num = G_centered.T @ y
den = np.sum(y**2)

beta = num / (den + 1e-8)

# residuals
y_pred = np.outer(y, beta)
residuals = G_centered - y_pred

sigma2 = np.sum(residuals**2, axis=0) / (len(y) - 2)

se_beta = np.sqrt(sigma2 / (den + 1e-8))

t_stat = beta / (se_beta + 1e-8)

pval = 2 * stats.t.sf(np.abs(t_stat), df=len(y)-2)

# ============================================================
# 7. FDR correction
# ============================================================
_, fdr, _, _ = multipletests(pval, method="fdr_bh")

# ============================================================
# 8. Rank genes
# ============================================================
order = np.argsort(fdr)

top_k = 20

print("\nTop genes (path-regression):")
for i in order[:top_k]:
    print(f"{genes_f[i]:20s}  beta={beta[i]:.3f}  FDR={fdr[i]:.3e}")

In [ ]:
plt.figure(figsize=(5, 4))

y = -np.log10(fdr)

plt.scatter(
    beta,
    y,
    s=6,
    alpha=0.3,
    color="grey"
)

# ------------------------------------------------------------
# Optional: set x limits here
# ------------------------------------------------------------
plt.xlim(-0.2, 0.2)   # <-- adjust as needed

# optional: cap y for readability
# y_cap = np.clip(y, 0, 50)

plt.xlabel("Effect size (beta)")
plt.ylabel("-log10(FDR)")
plt.title("Gene association with curvature")

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Prepare path
# ------------------------------------------------------------
p_ref = refined_paths
if p_ref.ndim == 3:
    p_ref = p_ref[0]

# ------------------------------------------------------------
# Arc length parameterization
# ------------------------------------------------------------
ds = np.linalg.norm(np.diff(p_ref, axis=0), axis=1)
s = np.concatenate([[0.0], np.cumsum(ds)])

# ------------------------------------------------------------
# Curvature (same as before)
# ------------------------------------------------------------
def curvature_along_path(path, X_emb, k_geod_c, epsilon=0.05):
    scale = np.mean(np.ptp(X_emb, axis=0))
    radius = epsilon * scale

    curv = np.empty(len(path))

    for i, p in enumerate(path):
        dists = np.linalg.norm(X_emb - p, axis=1)
        mask = dists <= radius
        curv[i] = np.mean(k_geod_c[mask]) if np.any(mask) else np.nan

    return curv


curv = curvature_along_path(p_ref, X_emb, k_geod_c, epsilon=0.05)

# interpolate NaNs
valid = np.isfinite(curv)
curv = np.interp(s, s[valid], curv[valid])

# ------------------------------------------------------------
# Proper cumulative curvature
# ------------------------------------------------------------
cum_curv = np.zeros_like(curv)
cum_curv[1:] = np.cumsum(
    0.5 * (curv[:-1] + curv[1:]) * ds
)

# ------------------------------------------------------------
# Project cells → arc-length coordinate
# ------------------------------------------------------------
def project_cells_to_path(X_emb, path, s_path):
    dists = np.linalg.norm(
        X_emb[:, None, :] - path[None, :, :],
        axis=2
    )
    idx = np.argmin(dists, axis=1)
    return s_path[idx]

s_cells = project_cells_to_path(X_emb, p_ref, s)

# ------------------------------------------------------------
# Kernel smoothing (cells → path)
# ------------------------------------------------------------
bandwidth = 0.05 * (s.max() - s.min())

diff = s[:, None] - s_cells[None, :]
W = np.exp(-0.5 * (diff / bandwidth)**2)
W /= (W.sum(axis=1, keepdims=True) + 1e-8)

# ------------------------------------------------------------
# Top genes
# ------------------------------------------------------------
top_k = 8
top_idx = order[:top_k]
top_genes = genes_f[top_idx]

# smooth gene curves
G = W @ Xf[:, top_idx]   # (path_points × genes)

# normalize genes (only)
G = (G - G.mean(axis=0)) / (G.std(axis=0) + 1e-8)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()

# gene curves
colors = ["#1f77b4", "#2ca02c", "#9467bd", "#17becf", "#e377c2"]

for i, g in enumerate(top_genes):
    ax1.plot(
        s,
        G[:, i],
        color=colors[i % len(colors)],
        linewidth=1.8,
        alpha=0.55,
        label=g
    )

ax1.set_xlabel("Arc length along path")
ax1.set_ylabel("Normalized gene expression")

# curvature (dotted)
ax2.plot(
    s,
    curv,
    color="#cc8c00",
    linestyle=":",
    linewidth=2.0,
    alpha=0.85,
    label="curvature"
)

# cumulative curvature (dominant)
ax2.plot(
    s,
    cum_curv,
    color="black",
    linewidth=3.0,
    alpha=0.95,
    label="cumulative curvature"
)

ax2.set_ylabel("Curvature / cumulative curvature")

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax2.spines["top"].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    frameon=False,
    fontsize=9,
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ============================================================
# 0. Prepare gene matrix
# ============================================================
X_gene = adata_raw.X
if hasattr(X_gene, "toarray"):
    X_gene = X_gene.toarray()

gene_names = adata_raw.var_names.to_numpy()

# ============================================================
# 1. Path geometry (arc length)
# ============================================================
p_ref = refined_paths
if p_ref.ndim == 3:
    p_ref = p_ref[0]

ds = np.linalg.norm(np.diff(p_ref, axis=0), axis=1)
s_path = np.concatenate([[0.0], np.cumsum(ds)])

# ============================================================
# 2. Project cells → path coordinate
# ============================================================
def project_cells_to_path(X_emb, path, s_path):
    dists = np.linalg.norm(
        X_emb[:, None, :] - path[None, :, :],
        axis=2
    )
    idx = np.argmin(dists, axis=1)
    return s_path[idx]

s_cells = project_cells_to_path(X_emb, p_ref, s_path)

# ============================================================
# 3. Filter genes
# ============================================================
mean_expr = X_gene.mean(axis=0)
var_expr  = X_gene.var(axis=0)

mask = (mean_expr > 0.05) & (var_expr > 0.01)

Xf = X_gene[:, mask]
genes_f = gene_names[mask]

print(f"Using {Xf.shape[1]} genes")

# ============================================================
# 4. Kernel smoothing (cells → path)
# ============================================================
bandwidth = 0.05 * (s_path.max() - s_path.min())

diff = s_path[:, None] - s_cells[None, :]
W = np.exp(-0.5 * (diff / bandwidth)**2)
W /= (W.sum(axis=1, keepdims=True) + 1e-8)

# smoothed gene curves (path_points × genes)
G = W @ Xf

# ============================================================
# 5. Prepare regression target (CUMULATIVE curvature on path)
# ============================================================

# --- interpolate curvature first ---
valid = np.isfinite(curv)
curv_smooth = np.interp(s_path, s_path[valid], curv[valid])

# --- proper cumulative curvature: ∫ κ ds ---
cum_curv = np.zeros_like(curv_smooth)
cum_curv[1:] = np.cumsum(
    0.5 * (curv_smooth[:-1] + curv_smooth[1:]) * ds
)

# --- standardize y ONLY ---
y = (cum_curv - cum_curv.mean()) / (cum_curv.std() + 1e-8)

# ============================================================
# 6. Linear regression along path (vectorized)
# ============================================================
G_centered = G - G.mean(axis=0)

num = G_centered.T @ y
den = np.sum(y**2)

beta = num / (den + 1e-8)

# residuals
y_pred = np.outer(y, beta)
residuals = G_centered - y_pred

sigma2 = np.sum(residuals**2, axis=0) / (len(y) - 2)

se_beta = np.sqrt(sigma2 / (den + 1e-8))

t_stat = beta / (se_beta + 1e-8)

pval = 2 * stats.t.sf(np.abs(t_stat), df=len(y)-2)

# ============================================================
# 7. FDR correction
# ============================================================
_, fdr, _, _ = multipletests(pval, method="fdr_bh")

# ============================================================
# 8. Rank genes
# ============================================================
order = np.argsort(fdr)

top_k = 20

print("\nTop genes (path-regression):")
for i in order[:top_k]:
    print(f"{genes_f[i]:20s}  beta={beta[i]:.3f}  FDR={fdr[i]:.3e}")

In [ ]:
plt.figure(figsize=(7, 4))

y = -np.log10(fdr)

# reference lines
plt.axvline(0, color="black", linewidth=1, alpha=0.4)
plt.axhline(0, linestyle="--", color="black", alpha=0.4)

# base scatter
plt.scatter(
    beta,
    y,
    s=1,
    alpha=0.15,
    color="grey"
)

# ------------------------------------------------------------
# Label top genes by effect size
# ------------------------------------------------------------
top_k = 15
idx = np.argsort(np.abs(beta))[-top_k:]

# highlight them
plt.scatter(beta[idx], y[idx], s=20, color="crimson", zorder=3)

# small offset to avoid overlap
xmin, xmax = beta.min(), beta.max()
dx = 0.01 * (xmax - xmin)
dy = 0.02 * (y.max() - y.min())

for i in idx:
    plt.text(
        beta[i] + np.sign(beta[i]) * dx,
        y[i] + dy,
        genes_f[i],
        fontsize=8,
        ha="center"
    )

# expand x-axis
pad = 0.1 * (xmax - xmin)
plt.xlim(xmin - pad, xmax + pad)

plt.xlabel("Effect size (beta)", fontsize=20)
plt.ylabel("-log10(FDR)", fontsize=20)
plt.title("Gene association with curvature (cumulative)", fontsize=24)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

y = -np.log10(fdr)

# reference lines
plt.axvline(0, color="black", linewidth=1, alpha=0.4)
plt.axhline(0, linestyle="--", color="black", alpha=0.4)

# base scatter
plt.scatter(
    beta,
    y,
    s=1,
    alpha=0.15,
    color="grey"
)

# ------------------------------------------------------------
# Custom gene labels
# ------------------------------------------------------------
genes_to_label = ["AFF3", "PRDM1", "IRF4", "GLCCI1", "CEP128", "FNDC3B"]

# map gene → index
gene_to_idx = {g: i for i, g in enumerate(genes_f)}

idx = [gene_to_idx[g] for g in genes_to_label if g in gene_to_idx]

# highlight points
plt.scatter(
    beta[idx],
    y[idx],
    s=30,
    color="crimson",
    zorder=3
)

# small offset to avoid overlap
xmin, xmax = beta.min(), beta.max()
dx = 0.01 * (xmax - xmin)
dy = 0.02 * (y.max() - y.min())

for g, i in zip(genes_to_label, idx):
    plt.text(
        beta[i] + np.sign(beta[i]) * dx,
        y[i] + dy,
        g,
        fontsize=10,
        ha="center"
    )

# expand x-axis
pad = 0.1 * (xmax - xmin)
plt.xlim(xmin - pad, xmax + pad)

plt.xlabel("Effect size (beta)", fontsize=20)
plt.ylabel("-log10(FDR)", fontsize=20)
plt.title("Gene association with curvature (cumulative)", fontsize=24)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(
    "./figures/bcell/volcano_curvature_genes.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Prepare path
# ------------------------------------------------------------
p_ref = refined_paths
if p_ref.ndim == 3:
    p_ref = p_ref[0]

# ------------------------------------------------------------
# Arc length parameterization (REAL units)
# ------------------------------------------------------------
ds = np.linalg.norm(np.diff(p_ref, axis=0), axis=1)
s = np.concatenate([[0.0], np.cumsum(ds)])

# normalized version ONLY for plotting
s_norm = (s - s.min()) / (s.max() - s.min() + 1e-8)

# ------------------------------------------------------------
# Project cells → arc-length (REAL units)
# ------------------------------------------------------------
dists = np.linalg.norm(
    X_emb[:, None, :] - p_ref[None, :, :],
    axis=2
)
s_cells = s[np.argmin(dists, axis=1)]   # <-- IMPORTANT: use s, not s_norm

# ------------------------------------------------------------
# Kernel smoothing (use REAL geometry)
# ------------------------------------------------------------
bandwidth = 0.05 * (s.max() - s.min())

diff = s[:, None] - s_cells[None, :]
W = np.exp(-0.5 * (diff / bandwidth)**2)
W /= (W.sum(axis=1, keepdims=True) + 1e-8)

# ------------------------------------------------------------
# Custom genes
# ------------------------------------------------------------
genes_to_plot = ["PRDM1", "IRF4", "GLCCI1", "CEP128", "FNDC3B"]

gene_to_idx = {g: i for i, g in enumerate(genes_f)}
idx = [gene_to_idx[g] for g in genes_to_plot if g in gene_to_idx]

# smooth gene curves
G = W @ Xf[:, idx]

# normalize genes
G = (G - G.mean(axis=0)) / (G.std(axis=0) + 1e-8)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()

colors = plt.cm.tab10.colors

# gene expression curves
for i, g in enumerate(genes_to_plot):
    if i >= G.shape[1]:
        continue
    ax1.plot(
        s_norm,   # <-- only here we use normalized axis
        G[:, i],
        color=colors[i % len(colors)],
        linewidth=2.0,
        alpha=0.7,
        label=g
    )

ax1.set_xlabel("Normalized path position", fontsize=20)
ax1.set_ylabel("Gene expression", fontsize=20)

# cumulative curvature
ax2.plot(
    s_norm,
    cum_curv,
    color="black",
    linewidth=3.0,
    alpha=0.95,
    label="cumulative curvature"
)

# optional curvature
if "curv" in locals():
    ax2.plot(
        s_norm,
        curv,
        color="#cc8c00",
        linestyle=":",
        linewidth=2.0,
        alpha=0.7,
        label="curvature"
    )

ax2.set_ylabel("Curvature", fontsize=20)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax2.spines["top"].set_visible(False)

# simple ticks
ax1.set_xticks(np.linspace(0, 1, 5))
ax1.set_yticks(np.arange(-2, 2.1, 1.0))
ax2.set_yticks([0, 1, 2, 3])

# legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    frameon=False,
    fontsize=15,
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    "./figures/bcell/cumulative_curvature_genes.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Normalize cell positions for plotting
# ------------------------------------------------------------
s_cells_norm = (s_cells - s.min()) / (s.max() - s.min() + 1e-8)

# ------------------------------------------------------------
# Custom genes
# ------------------------------------------------------------
genes_to_plot = ["PRDM1", "IRF4", "GLCCI1", "CEP128", "FNDC3B"]

gene_to_idx = {g: i for i, g in enumerate(genes_f)}
idx = [gene_to_idx[g] for g in genes_to_plot if g in gene_to_idx]

# ------------------------------------------------------------
# Plot: 1 x 5 grid
# ------------------------------------------------------------
fig, axes = plt.subplots(1, len(idx), figsize=(15, 3), sharex=True, sharey=True)

if len(idx) == 1:
    axes = [axes]

for ax, g, i in zip(axes, genes_to_plot, idx):

    expr = Xf[:, i]

    # optional: light normalization for visibility
    expr = (expr - np.mean(expr)) / (np.std(expr) + 1e-8)

    ax.scatter(
        s_cells_norm,
        expr,
        s=1,
        alpha=0.2,
        color="black"
    )

    ax.set_title(g, fontsize=12)

    # clean look
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # simple ticks
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([-2, 0, 2])

# labels only on edges
axes[0].set_ylabel("Expression", fontsize=12)
for ax in axes:
    ax.set_xlabel("Position", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Custom genes
# ------------------------------------------------------------
genes_to_plot = ["PRDM1", "IRF4", "GLCCI1", "CEP128", "FNDC3B"]

gene_to_idx = {g: i for i, g in enumerate(genes_f)}
idx = [gene_to_idx[g] for g in genes_to_plot if g in gene_to_idx]

# ------------------------------------------------------------
# Plot: 1 x 5 grid
# ------------------------------------------------------------
fig, axes = plt.subplots(1, len(idx), figsize=(15, 3))

if len(idx) == 1:
    axes = [axes]

for ax, g, i in zip(axes, genes_to_plot, idx):

    expr = Xf[:, i]

    # optional normalization (recommended for visualization)
    expr = (expr - np.mean(expr)) / (np.std(expr) + 1e-8)

    sc = ax.scatter(
        X_emb[:, 0],
        X_emb[:, 1],
        c=expr,
        s=2,
        cmap="viridis",
        alpha=0.8
    )

    ax.set_title(g, fontsize=12)

    # clean embedding look
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

# ------------------------------------------------------------
# shared colorbar
# ------------------------------------------------------------
# cbar = fig.colorbar(sc, ax=axes, fraction=0.02, pad=0.02)
cbar.set_label("Expression", fontsize=12)

plt.tight_layout()
plt.show()